# Load DBU Multipliers

This notebook creates the `dbu_multipliers` table containing:
1. **Photon multipliers** - varies by cloud and SKU type
2. **Serverless feature multipliers** - for Automated/Interactive Serverless SKUs

**Sources:**
- AWS: https://docs.databricks.com/aws/en/resources/pricing
- Azure: https://learn.microsoft.com/en-gb/azure/databricks/resources/pricing
- GCP: https://docs.databricks.com/gcp/en/resources/pricing


In [ ]:
# Configuration
CATALOG = "lakemeter_catalog"
SCHEMA = "lakemeter"
TABLE_DBU_MULTIPLIERS = f"{CATALOG}.{SCHEMA}.dbu_multipliers"

print(f"✅ Target table: {TABLE_DBU_MULTIPLIERS}")


In [ ]:
import pandas as pd
from datetime import datetime

# =============================================================================
# PHOTON MULTIPLIERS - Classic Compute (varies by cloud)
# =============================================================================

# Photon multipliers for Jobs compute (DLT follows Jobs)
PHOTON_JOBS = {
    "AWS": 2.9,
    "AZURE": 2.5,
    "GCP": 2.5,
}

# Photon multipliers for All-Purpose compute
PHOTON_ALL_PURPOSE = {
    "AWS": 2.0,
    "AZURE": 2.0,
    "GCP": 2.0,
}

# SKU types that use Jobs Photon multiplier
JOBS_SKU_TYPES = [
    "JOBS_COMPUTE",
    "DLT_CORE_COMPUTE",
    "DLT_PRO_COMPUTE",
    "DLT_ADVANCED_COMPUTE",
]

# SKU types that use All-Purpose Photon multiplier
ALL_PURPOSE_SKU_TYPES = [
    "ALL_PURPOSE_COMPUTE",
]

# Build Photon multiplier rows
photon_rows = []

# Jobs/DLT Photon
for cloud, multiplier in PHOTON_JOBS.items():
    for sku_type in JOBS_SKU_TYPES:
        photon_rows.append({
            "cloud": cloud,
            "sku_type": sku_type,
            "feature": "photon",
            "multiplier": multiplier,
            "category": "CLASSIC_COMPUTE",
        })

# All-Purpose Photon
for cloud, multiplier in PHOTON_ALL_PURPOSE.items():
    for sku_type in ALL_PURPOSE_SKU_TYPES:
        photon_rows.append({
            "cloud": cloud,
            "sku_type": sku_type,
            "feature": "photon",
            "multiplier": multiplier,
            "category": "CLASSIC_COMPUTE",
        })

print(f"✅ Photon multipliers: {len(photon_rows)} rows")


In [ ]:
# =============================================================================
# SERVERLESS FEATURE MULTIPLIERS (currently same across clouds)
# =============================================================================

CLOUDS = ["AWS", "AZURE", "GCP"]

# Automated Serverless SKU features
AUTOMATED_SERVERLESS_FEATURES = [
    {"feature": "serverless_jobs", "multiplier": 1.0},
    {"feature": "serverless_dlt", "multiplier": 1.0},
    {"feature": "predictive_optimization", "multiplier": 1.0},
    {"feature": "data_quality_monitoring", "multiplier": 2.0},
    {"feature": "fine_grained_access_control", "multiplier": 1.0},
    {"feature": "online_tables_sync", "multiplier": 1.0},
    {"feature": "online_tables_capacity", "multiplier": 2.0},
    {"feature": "materialized_views_streaming_tables", "multiplier": 1.0},
    {"feature": "data_classification", "multiplier": 3.0},
]

# Interactive Serverless SKU features
INTERACTIVE_SERVERLESS_FEATURES = [
    {"feature": "serverless_notebook", "multiplier": 1.0},
    {"feature": "databricks_app", "multiplier": 0.5},
]

# Database Serverless (Lakebase)
DATABASE_SERVERLESS_FEATURES = [
    {"feature": "lakebase", "multiplier": 1.0},
]

# Build serverless feature rows
serverless_rows = []

# Automated Serverless
for cloud in CLOUDS:
    for f in AUTOMATED_SERVERLESS_FEATURES:
        serverless_rows.append({
            "cloud": cloud,
            "sku_type": "JOBS_SERVERLESS_COMPUTE",
            "feature": f["feature"],
            "multiplier": f["multiplier"],
            "category": "AUTOMATED_SERVERLESS",
        })

# Interactive Serverless
for cloud in CLOUDS:
    for f in INTERACTIVE_SERVERLESS_FEATURES:
        serverless_rows.append({
            "cloud": cloud,
            "sku_type": "ALL_PURPOSE_SERVERLESS_COMPUTE",
            "feature": f["feature"],
            "multiplier": f["multiplier"],
            "category": "INTERACTIVE_SERVERLESS",
        })

# Database Serverless
for cloud in CLOUDS:
    for f in DATABASE_SERVERLESS_FEATURES:
        serverless_rows.append({
            "cloud": cloud,
            "sku_type": "DATABASE_SERVERLESS_COMPUTE",
            "feature": f["feature"],
            "multiplier": f["multiplier"],
            "category": "DATABASE_SERVERLESS",
        })

print(f"✅ Serverless feature multipliers: {len(serverless_rows)} rows")


In [ ]:
# =============================================================================
# COMBINE AND CREATE DATAFRAME
# =============================================================================

all_rows = photon_rows + serverless_rows

# Add metadata
for row in all_rows:
    row["updated_at"] = datetime.utcnow().isoformat()

df_multipliers = pd.DataFrame(all_rows)

print(f"\n📊 Total multipliers: {len(df_multipliers)} rows")
print(f"\n📊 By category:")
print(df_multipliers['category'].value_counts())

print(f"\n📊 By feature:")
print(df_multipliers['feature'].value_counts())

# Show all data
print(f"\n📊 Full table:")
display(df_multipliers)


In [ ]:
# =============================================================================
# SAVE TO TABLE
# =============================================================================

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(df_multipliers)

# Save to table
spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABLE_DBU_MULTIPLIERS)

print(f"✅ Saved {len(df_multipliers)} multipliers to {TABLE_DBU_MULTIPLIERS}")


In [ ]:
# =============================================================================
# VERIFY
# =============================================================================

print("📊 Photon Multipliers by Cloud:")
display(spark.sql(f"""
    SELECT cloud, sku_type, multiplier
    FROM {TABLE_DBU_MULTIPLIERS}
    WHERE feature = 'photon'
    ORDER BY cloud, sku_type
"""))

print("\n📊 Serverless Feature Multipliers (non-1X):")
display(spark.sql(f"""
    SELECT feature, multiplier, category
    FROM {TABLE_DBU_MULTIPLIERS}
    WHERE multiplier != 1.0 AND cloud = 'AWS'
    ORDER BY multiplier DESC
"""))

print("\n📊 All Multipliers:")
display(spark.sql(f"""
    SELECT cloud, sku_type, feature, multiplier, category
    FROM {TABLE_DBU_MULTIPLIERS}
    ORDER BY category, cloud, feature
"""))
